In [23]:
import pyautogui as pyg
pyg.mouseInfo()


In [2]:
import pyautogui as pyg
pyg.sleep(2)
basex=327
basey=384
gap=40

In [3]:
def main():

    print("寶石方塊自動消除啟動")
    print("請先進入遊戲並確認棋盤位置")
    print("按 Q 可以停止")
    print("移動滑鼠到左上角也可以緊急停止")

    start_time = time.time()
    moves = 0
    no_move_count = 0

    while True:

        # 手動停止
        if keyboard.is_pressed("q"):
            print("使用者停止")
            break

        # 時間停止
        if time.time() - start_time >= MAX_RUNTIME:
            print("執行時間結束")
            break

        # 讀取真實螢幕
        img = capture_board()

        board = read_board(img)

        # 尋找最佳消除
        move, score = find_best_move(board)

        if move is None:

            no_move_count += 1

            print("目前沒有找到可消除組合")

            if no_move_count >= 8:
                print("連續無法找到有效移動，停止")
                break

            time.sleep(0.4)
            continue

        no_move_count = 0

        print(
            f"第 {moves+1} 次交換 | "
            f"位置：{move} | "
            f"消除分數：{score}"
        )

        # 執行交換
        perform_move(move)

        moves += 1

        # 等待寶石消除及掉落
        time.sleep(WAIT_TIME)

    print("====================")
    print("自動消除結束")
    print("總交換次數：", moves)


if __name__ == "__main__":
    main()

寶石方塊自動消除啟動
請先進入遊戲並確認棋盤位置
按 Q 可以停止
移動滑鼠到左上角也可以緊急停止


NameError: name 'time' is not defined

In [57]:
import ctypes
import sys
import time

import cv2
import numpy as np
import pyautogui
from PIL import ImageGrab

# 讓 pyautogui 與截圖使用相同的實體像素座標（高 DPI 螢幕必備）
try:
    ctypes.windll.shcore.SetProcessDpiAwareness(1)
except Exception:
    pass

# =========================
# 棋盤位置（螢幕座標）
# =========================
LEFT, TOP = 387,507 
RIGHT, BOTTOM = 775, 896
WIDTH, HEIGHT = RIGHT - LEFT, BOTTOM - TOP
N = 8

CW, CH = WIDTH / N, HEIGHT / N

# =========================
# 寶石定位
# =========================
# basex, basey：左上角寶石的中心點（螢幕座標）
basex = int(LEFT + CW / 2)
basey = int(TOP + CH / 2)

# gap：每顆寶石的間距（可給單一數值，或 (水平, 垂直)）
gap = (CW, CH)


def gem_pos(r, c):
    """回傳第 r 列（row）、第 c 行（column）那顆寶石的中心點 (x, y)。"""
    gap_x, gap_y = gap if isinstance(gap, (tuple, list)) else (gap, gap)
    return int(basex + c * gap_x), int(basey + r * gap_y)


# =========================
# 時間與流程設定
# =========================
ROUND_TIME = 61 # Bejeweled Blitz 一局約 60 秒，留一點緩衝
START_DELAY = 1 # 按下執行後，等幾秒才開始動作
STILL_LIMIT = 8 # 連續幾次找不到步可走，就判定遊戲結束
SETTLE_TIMEOUT = 1 # 等棋盤穩定的最長時間（秒）
VERBOSE = True # 是否印出每次辨識到的棋盤

# 不列入消除判斷的格子
IGNORE = {"empty", "unknown"}

SYMBOL = {
    "red": "R", "orange": "O", "yellow": "Y", "green": "G",
    "blue": "B", "purple": "P", "white": "W",
    "empty": ".", "unknown": "?",
}


# =========================
# 截圖
# =========================
def capture():
    img = ImageGrab.grab(
        bbox=(LEFT, TOP, RIGHT, BOTTOM), all_screens=True
    )
    return cv2.cvtColor(np.array(img), cv2.COLOR_RGB2BGR)


def stop_pressed():
    """按 Q 停止（不需額外套件）。"""
    return bool(ctypes.windll.user32.GetAsyncKeyState(ord("Q")) & 0x8000)


# =========================
# 顏色辨識
# =========================
def identify(patch):
    """把一小塊畫面（BGR）轉成寶石顏色名稱。"""
    hsv = cv2.cvtColor(patch, cv2.COLOR_BGR2HSV)
    h, s, v = (int(x) for x in np.median(hsv.reshape(-1, 3), axis=0))

    if v < 55: # 太暗 → 空格
        return "empty"
    if s < 45: # 幾乎沒有飽和度 → 白色 / 特殊寶石
        return "white"
    if h < 8 or h >= 170:
        return "red"
    if h < 22:
        return "orange"
    if h < 35:
        return "yellow"
    if h < 85:
        return "green"
    if h < 115:
        return "blue"
    return "purple"


def read_board(img):
    """把截圖轉成 8x8 的顏色盤面。"""
    board = []
    for r in range(N):
        row = []
        for c in range(N):
            x = int((c + .5) * CW)
            y = int((r + .5) * CH)
            patch = img[y - 4:y + 5, x - 4:x + 5]
            row.append(identify(patch))
        board.append(row)
    return board


def print_board(board):
    for row in board:
        print(" " + " ".join(SYMBOL.get(c, "?") for c in row))


# =========================
# 消除判斷
# =========================
def find_matches(board):
    """回傳所有會被消除的格子座標集合 {(r, c), ...}。"""
    found = set()

    # 水平
    for r in range(N):
        c = 0
        while c < N:
            color = board[r][c]
            end = c
            while end < N and board[r][end] == color:
                end += 1
            if color not in IGNORE and end - c >= 3:
                found.update((r, x) for x in range(c, end))
            c = end

    # 垂直
    for c in range(N):
        r = 0
        while r < N:
            color = board[r][c]
            end = r
            while end < N and board[end][c] == color:
                end += 1
            if color not in IGNORE and end - r >= 3:
                found.update((y, c) for y in range(r, end))
            r = end

    return found


# 舊名稱，方便沿用
matches = find_matches


def best_move(board):
    """回傳 (a, b, score)；找不到就回傳 None。"""
    best = None
    best_score = 0

    for r in range(N):
        for c in range(N):
            for dr, dc in ((0, 1), (1, 0)):
                r2, c2 = r + dr, c + dc
                if r2 >= N or c2 >= N:
                    continue

                color = board[r][c]
                other = board[r2][c2]
                if color in IGNORE or other in IGNORE:
                    continue
                if color == other:
                    continue

                test = [row[:] for row in board]
                test[r][c], test[r2][c2] = other, color

                found = find_matches(test)

                # 交換的位置必須真的被消掉，才算是有效步
                if (r, c) not in found and (r2, c2) not in found:
                    continue

                score = len(found)
                if score > best_score:
                    best_score = score
                    best = ((r, c), (r2, c2), score)

    return best


def board_is_empty(board):
    """整個盤面幾乎都是空格 → 遊戲結束（或已離開遊戲）。"""
    empty = sum(cell == "empty" for row in board for cell in row)
    return empty >= N * N * 0.7


def signature(board):
    return tuple(tuple(row) for row in board)


# =========================
# 滑鼠交換
# =========================
def swap(a, b):
    x1, y1 = gem_pos(*a)
    x2, y2 = gem_pos(*b)
    pyautogui.moveTo(x1, y1)
    pyautogui.dragTo(x2, y2, duration=0.12, button="left")


def wait_stable(timeout=SETTLE_TIMEOUT):
    """等畫面連續兩次一樣，代表掉落動畫結束。"""
    deadline = time.time() + timeout
    last = None
    same = 0
    board = None

    while time.time() < deadline:
        board = read_board(capture())
        sig = signature(board)
        if sig == last:
            same += 1
            if same >= 2:
                return board
        else:
            same = 0
            last = sig
        time.sleep(0.06)

    return board


# =========================
# 校正模式
# =========================
def calibrate():
    img = capture()
    board = read_board(img)
    hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)

    print(f"棋盤範圍：x {LEFT}~{RIGHT}, y {TOP}~{BOTTOM}（{WIDTH}x{HEIGHT}）")
    print("辨識結果（R 紅 O 橙 Y 黃 G 綠 B 藍 P 紫 W 白 . 空）：")
    print_board(board)

    print("\n各格中心 HSV（h, s, v）：")
    for r in range(N):
        for c in range(N):
            x = int((c + .5) * CW)
            y = int((r + .5) * CH)
            h, s, v = (int(n) for n in np.median(
                hsv[y - 4:y + 5, x - 4:x + 5].reshape(-1, 3), axis=0))
            print(f" ({r},{c}) -> {board[r][c]:7s} h={h:3d} s={s:3d} v={v:3d}")

    path = "board_debug.png"
    ok, buf = cv2.imencode(".png", img)
    if ok:
        buf.tofile(path) # 用 tofile 才支援中文路徑
        print(f"\n截圖已存成 {path}")


# =========================
# 邏輯測試（不需要遊戲畫面）
# =========================
def selftest():
    import random

    rng = random.Random(20260101)
    colors = ["red", "green", "blue", "yellow", "purple", "orange"]

    # 固定亂數種子，找一個「沒有現成三連、但存在有效步」的盤面
    base = None
    for _ in range(5000):
        cand = [[rng.choice(colors) for _ in range(N)] for _ in range(N)]
        if not find_matches(cand) and best_move(cand) is not None:
            base = cand
            break
    assert base is not None, "找不到合適的測試盤面"

    # 橫向三連要被找到
    lined = [row[:] for row in base]
    lined[0][0] = lined[0][1] = lined[0][2] = "red"
    found = find_matches(lined)
    assert {(0, 0), (0, 1), (0, 2)} <= found, "橫向三連沒抓到"

    # 縱向三連要被找到
    lined = [row[:] for row in base]
    lined[0][4] = lined[1][4] = lined[2][4] = "blue"
    found = find_matches(lined)
    assert {(0, 4), (1, 4), (2, 4)} <= found, "縱向三連沒抓到"

    # 一定找得到有效步，而且該步真的能消
    move = best_move(base)
    assert move is not None, "找不到有效步"

    a, b, score = move
    test = [row[:] for row in base]
    test[a[0]][a[1]], test[b[0]][b[1]] = test[b[0]][b[1]], test[a[0]][a[1]]
    found = find_matches(test)
    assert a in found or b in found, "選到的步不會消除"
    assert score == len(found), "分數與實際消除數不一致"

    # 空格過多 → 視為遊戲結束
    empty = [["empty"] * N for _ in range(N)]
    assert board_is_empty(empty)

    print(f"selftest OK（範例最佳步：{a} <-> {b}，可消 {score} 顆）")


# =========================
# 主流程
# =========================
def main():
    print("寶石方塊 自動消除")
    print(f"棋盤範圍：x {LEFT}~{RIGHT}, y {TOP}~{BOTTOM}")
    print(f"{START_DELAY} 秒後開始，請先切換到遊戲畫面")
    print("按 q，或把滑鼠移到左上角可停止")
    time.sleep(START_DELAY)

    start = time.time()
    moves = 0
    still = 0
    last_board = None

    while True:
        if stop_pressed():
            print("手動停止")
            break

        elapsed = time.time() - start
        if elapsed >= ROUND_TIME:
            print(f"時間到（{elapsed:.1f} 秒），停止")
            break

        board = read_board(capture())

        if board_is_empty(board):
            print("盤面幾乎空了，遊戲可能已結束")
            break

        if VERBOSE and board != last_board:
            print_board(board)
            last_board = [row[:] for row in board]

        move = best_move(board)

        if move is None:
            still += 1
            print(f"找不到可消除的組合（{still}/{STILL_LIMIT}）")
            if still >= STILL_LIMIT:
                print("連續找不到步可走，判定遊戲結束")
                break
            time.sleep(0.4)
            continue

        still = 0
        a, b, score = move
        print(f"[{elapsed:5.1f}s] 交換 {a} <-> {b}，預計消除 {score} 顆")
        swap(a, b)
        moves += 1
        wait_stable()

    print(f"結束：共交換 {moves} 次，耗時 {time.time() - start:.1f} 秒")


if __name__ == "__main__":
    mode = sys.argv[1] if len(sys.argv) > 1 else "run"
    if mode == "calibrate":
        calibrate()
    elif mode == "selftest":
        selftest()
    else:
        main()


import ctypes
import sys
import time

import cv2
import numpy as np
import pyautogui
from PIL import ImageGrab

# 讓 pyautogui 與截圖使用相同的實體像素座標（高 DPI 螢幕必備）
try:
    ctypes.windll.shcore.SetProcessDpiAwareness(1)
except Exception:
    pass

# =========================
# 棋盤位置（螢幕座標）
# =========================
LEFT, TOP = 495, 371
RIGHT, BOTTOM = 820, 691
WIDTH, HEIGHT = RIGHT - LEFT, BOTTOM - TOP
N = 8

CW, CH = WIDTH / N, HEIGHT / N

# =========================
# 寶石定位
# =========================
# basex, basey：左上角寶石的中心點（螢幕座標）
basex = int(LEFT + CW / 2)
basey = int(TOP + CH / 2)

# gap：每顆寶石的間距（可給單一數值，或 (水平, 垂直)）
gap = (CW, CH)


def gem_pos(r, c):
    """回傳第 r 列（row）、第 c 行（column）那顆寶石的中心點 (x, y)。"""
    gap_x, gap_y = gap if isinstance(gap, (tuple, list)) else (gap, gap)
    return int(basex + c * gap_x), int(basey + r * gap_y)


# =========================
# 時間與流程設定
# =========================
ROUND_TIME = 62 # Bejeweled Blitz 一局約 60 秒，留一點緩衝
START_DELAY = 1 # 按下執行後，等幾秒才開始動作
STILL_LIMIT = 10 # 連續幾次找不到步可走，就判定遊戲結束
SETTLE_TIMEOUT = 2.0 # 等棋盤穩定的最長時間（秒）
VERBOSE = True # 是否印出每次辨識到的棋盤

# 不列入消除判斷的格子
IGNORE = {"empty", "unknown"}

SYMBOL = {
    "red": "R", "orange": "O", "yellow": "Y", "green": "G",
    "blue": "B", "purple": "P", "white": "W",
    "empty": ".", "unknown": "?",
}


# =========================
# 截圖
# =========================
def capture():
    img = ImageGrab.grab(
        bbox=(LEFT, TOP, RIGHT, BOTTOM), all_screens=True
    )
    return cv2.cvtColor(np.array(img), cv2.COLOR_RGB2BGR)


def stop_pressed():
    """按 Q 停止（不需額外套件）。"""
    return bool(ctypes.windll.user32.GetAsyncKeyState(ord("Q")) & 0x8000)


# =========================
# 顏色辨識
# =========================
def identify(patch):
    """把一小塊畫面（BGR）轉成寶石顏色名稱。"""
    hsv = cv2.cvtColor(patch, cv2.COLOR_BGR2HSV)
    h, s, v = (int(x) for x in np.median(hsv.reshape(-1, 3), axis=0))

    if v < 55: # 太暗 → 空格
        return "empty"
    if s < 45: # 幾乎沒有飽和度 → 白色 / 特殊寶石
        return "white"
    if h < 8 or h >= 170:
        return "red"
    if h < 22:
        return "orange"
    if h < 35:
        return "yellow"
    if h < 85:
        return "green"
    if h < 115:
        return "blue"
    return "purple"


def read_board(img):
    """把截圖轉成 8x8 的顏色盤面。"""
    board = []
    for r in range(N):
        row = []
        for c in range(N):
            x = int((c + .5) * CW)
            y = int((r + .5) * CH)
            patch = img[y - 4:y + 5, x - 4:x + 5]
            row.append(identify(patch))
        board.append(row)
    return board


def print_board(board):
    for row in board:
        print(" " + " ".join(SYMBOL.get(c, "?") for c in row))


# =========================
# 消除判斷
# =========================
def find_matches(board):
    """回傳所有會被消除的格子座標集合 {(r, c), ...}。"""
    found = set()

    # 水平
    for r in range(N):
        c = 0
        while c < N:
            color = board[r][c]
            end = c
            while end < N and board[r][end] == color:
                end += 1
            if color not in IGNORE and end - c >= 3:
                found.update((r, x) for x in range(c, end))
            c = end

    # 垂直
    for c in range(N):
        r = 0
        while r < N:
            color = board[r][c]
            end = r
            while end < N and board[end][c] == color:
                end += 1
            if color not in IGNORE and end - r >= 3:
                found.update((y, c) for y in range(r, end))
            r = end

    return found


# 舊名稱，方便沿用
matches = find_matches


def best_move(board):
    """回傳 (a, b, score)；找不到就回傳 None。"""
    best = None
    best_score = 0

    for r in range(N):
        for c in range(N):
            for dr, dc in ((0, 1), (1, 0)):
                r2, c2 = r + dr, c + dc
                if r2 >= N or c2 >= N:
                    continue

                color = board[r][c]
                other = board[r2][c2]
                if color in IGNORE or other in IGNORE:
                    continue
                if color == other:
                    continue

                test = [row[:] for row in board]
                test[r][c], test[r2][c2] = other, color

                found = find_matches(test)

                # 交換的位置必須真的被消掉，才算是有效步
                if (r, c) not in found and (r2, c2) not in found:
                    continue

                score = len(found)
                if score > best_score:
                    best_score = score
                    best = ((r, c), (r2, c2), score)

    return best


def board_is_empty(board):
    """整個盤面幾乎都是空格 → 遊戲結束（或已離開遊戲）。"""
    empty = sum(cell == "empty" for row in board for cell in row)
    return empty >= N * N * 0.7


def signature(board):
    return tuple(tuple(row) for row in board)


# =========================
# 滑鼠交換
# =========================
def swap(a, b):
    x1, y1 = gem_pos(*a)
    x2, y2 = gem_pos(*b)
    pyautogui.moveTo(x1, y1)
    pyautogui.dragTo(x2, y2, duration=0.12, button="left")


def wait_stable(timeout=SETTLE_TIMEOUT):
    """等畫面連續兩次一樣，代表掉落動畫結束。"""
    deadline = time.time() + timeout
    last = None
    same = 0
    board = None

    while time.time() < deadline:
        board = read_board(capture())
        sig = signature(board)
        if sig == last:
            same += 1
            if same >= 2:
                return board
        else:
            same = 0
            last = sig
        time.sleep(0.06)

    return board


# =========================
# 校正模式
# =========================
def calibrate():
    img = capture()
    board = read_board(img)
    hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)

    print(f"棋盤範圍：x {LEFT}~{RIGHT}, y {TOP}~{BOTTOM}（{WIDTH}x{HEIGHT}）")
    print("辨識結果（R 紅 O 橙 Y 黃 G 綠 B 藍 P 紫 W 白 . 空）：")
    print_board(board)

    print("\n各格中心 HSV（h, s, v）：")
    for r in range(N):
        for c in range(N):
            x = int((c + .5) * CW)
            y = int((r + .5) * CH)
            h, s, v = (int(n) for n in np.median(
                hsv[y - 4:y + 5, x - 4:x + 5].reshape(-1, 3), axis=0))
            print(f" ({r},{c}) -> {board[r][c]:7s} h={h:3d} s={s:3d} v={v:3d}")

    path = "board_debug.png"
    ok, buf = cv2.imencode(".png", img)
    if ok:
        buf.tofile(path) # 用 tofile 才支援中文路徑
        print(f"\n截圖已存成 {path}")


# =========================
# 邏輯測試（不需要遊戲畫面）
# =========================
def selftest():
    import random

    rng = random.Random(20260101)
    colors = ["red", "green", "blue", "yellow", "purple", "orange"]

    # 固定亂數種子，找一個「沒有現成三連、但存在有效步」的盤面
    base = None
    for _ in range(5000):
        cand = [[rng.choice(colors) for _ in range(N)] for _ in range(N)]
        if not find_matches(cand) and best_move(cand) is not None:
            base = cand
            break
    assert base is not None, "找不到合適的測試盤面"

    # 橫向三連要被找到
    lined = [row[:] for row in base]
    lined[0][0] = lined[0][1] = lined[0][2] = "red"
    found = find_matches(lined)
    assert {(0, 0), (0, 1), (0, 2)} <= found, "橫向三連沒抓到"

    # 縱向三連要被找到
    lined = [row[:] for row in base]
    lined[0][4] = lined[1][4] = lined[2][4] = "blue"
    found = find_matches(lined)
    assert {(0, 4), (1, 4), (2, 4)} <= found, "縱向三連沒抓到"

    # 一定找得到有效步，而且該步真的能消
    move = best_move(base)
    assert move is not None, "找不到有效步"

    a, b, score = move
    test = [row[:] for row in base]
    test[a[0]][a[1]], test[b[0]][b[1]] = test[b[0]][b[1]], test[a[0]][a[1]]
    found = find_matches(test)
    assert a in found or b in found, "選到的步不會消除"
    assert score == len(found), "分數與實際消除數不一致"

    # 空格過多 → 視為遊戲結束
    empty = [["empty"] * N for _ in range(N)]
    assert board_is_empty(empty)

    print(f"selftest OK（範例最佳步：{a} <-> {b}，可消 {score} 顆）")


# =========================
# 主流程
# =========================
def main():
    print("寶石方塊 自動消除")
    print(f"棋盤範圍：x {LEFT}~{RIGHT}, y {TOP}~{BOTTOM}")
    print(f"{START_DELAY} 秒後開始，請先切換到遊戲畫面")
    print("按 q，或把滑鼠移到左上角可停止")
    time.sleep(START_DELAY)

    start = time.time()
    moves = 0
    still = 0
    last_board = None

    while True:
        if stop_pressed():
            print("手動停止")
            break

        elapsed = time.time() - start
        if elapsed >= ROUND_TIME:
            print(f"時間到（{elapsed:.1f} 秒），停止")
            break

        board = read_board(capture())

        if board_is_empty(board):
            print("盤面幾乎空了，遊戲可能已結束")
            break

        if VERBOSE and board != last_board:
            print_board(board)
            last_board = [row[:] for row in board]

        move = best_move(board)

        if move is None:
            still += 1
            print(f"找不到可消除的組合（{still}/{STILL_LIMIT}）")
            if still >= STILL_LIMIT:
                print("連續找不到步可走，判定遊戲結束")
                break
            time.sleep(0.4)
            continue

        still = 0
        a, b, score = move
        print(f"[{elapsed:5.1f}s] 交換 {a} <-> {b}，預計消除 {score} 顆")
        swap(a, b)
        moves += 1
        wait_stable()

    print(f"結束：共交換 {moves} 次，耗時 {time.time() - start:.1f} 秒")


if __name__ == "__main__":
    mode = sys.argv[1] if len(sys.argv) > 1 else "run"
    if mode == "calibrate":
        calibrate()
    elif mode == "selftest":
        selftest()
    else:
        main()


寶石方塊 自動消除
棋盤範圍：x 387~775, y 507~896
1 秒後開始，請先切換到遊戲畫面
按 q，或把滑鼠移到左上角可停止
 W W G W P B B R
 W G R W R W B G
 Y G G Y G Y R G
 B Y P R W W O B
 R P Y G P G B G
 G P R Y B B W B
 B Y Y W P O G B
 W W P Y R W O O
[  0.0s] 交換 (3, 7) <-> (4, 7)，預計消除 6 顆
 W W G W P B B .
 W G R W R W B .
 Y G G Y G Y R .
 B Y P R W W O .
 R P Y G P G B W
 G P R Y B B W O
 B Y Y W P O G R
 W W P Y R W O O
[  1.5s] 交換 (0, 1) <-> (0, 2)，預計消除 3 顆
 W . W W P B B Y
 W . R W R W B O
 Y . G Y G Y R P
 B Y P R W W O .
 R P Y G P G B W
 G P R Y B B W O
 B Y Y W P O G R
 W W P Y R W O O
[  2.6s] 交換 (3, 1) <-> (3, 2)，預計消除 3 顆
 W . W W P B B Y
 W . R W R W B O
 Y R G Y G Y R P
 B Y Y R W W O .
 R P Y G P G B W
 G P R Y B B W O
 B Y Y W P O G R
 W W P Y R W O O
[  3.6s] 交換 (5, 2) <-> (5, 3)，預計消除 4 顆
 W R W W P B B Y
 W Y R W R W B O
 Y B G Y G Y R P
 B Y Y R W W O .
 R R Y G P G B W
 G Y R Y B B W O
 B Y Y W P O G R
 W W P Y R W O O
[  4.7s] 交換 (4, 2) <-> (5, 2)，預計消除 6 顆
 Y Y Y W P B B Y
 W . W W R W B O
 W G R Y G Y R P
 Y R

FailSafeException: PyAutoGUI fail-safe triggered from mouse moving to a corner of the screen. To disable this fail-safe, set pyautogui.FAILSAFE to False. DISABLING FAIL-SAFE IS NOT RECOMMENDED.

In [10]:
import ctypes
import sys
import time
import cv2
import numpy as np
import pyautogui
from PIL import ImageGrab

# 高 DPI 螢幕設定
try:
    ctypes.windll.shcore.SetProcessDpiAwareness(1)
except Exception:
    pass

# =========================
# 遊戲與棋盤設定
# =========================
# 提示：若要在不同視窗位置使用，可自行調整此處座標
LEFT, TOP, RIGHT, BOTTOM = 495, 371, 820, 691
WIDTH, HEIGHT = RIGHT - LEFT, BOTTOM - TOP
N = 8
CW, CH = WIDTH / N, HEIGHT / N

BASEX = int(LEFT + CW / 2)
BASEY = int(TOP + CH / 2)

ROUND_TIME = 62
START_DELAY = 5
STILL_LIMIT = 8
SETTLE_TIMEOUT = 2.0
VERBOSE = True

IGNORE = {"empty", "unknown"}
SYMBOL = {
    "red": "R", "orange": "O", "yellow": "Y", "green": "G",
    "blue": "B", "purple": "P", "white": "W", "empty": ".", "unknown": "?"
}

# =========================
# 核心功能：截圖與控制
# =========================
def capture():
    img = ImageGrab.grab(bbox=(LEFT, TOP, RIGHT, BOTTOM), all_screens=True)
    return cv2.cvtColor(np.array(img), cv2.COLOR_RGB2BGR)

def stop_pressed():
    return bool(ctypes.windll.user32.GetAsyncKeyState(ord("Q")) & 0x8000)

def gem_pos(r, c):
    return int(BASEX + c * CW), int(BASEY + r * CH)

def swap(a, b):
    x1, y1 = gem_pos(*a)
    x2, y2 = gem_pos(*b)
    pyautogui.moveTo(x1, y1)
    pyautogui.dragTo(x2, y2, duration=0.1, button="left")

# =========================
# 影像辨識
# =========================
def identify(patch):
    hsv = cv2.cvtColor(patch, cv2.COLOR_BGR2HSV)
    h, s, v = (int(x) for x in np.median(hsv.reshape(-1, 3), axis=0))
    if v < 55: return "empty"
    if s < 45: return "white"
    if h < 8 or h >= 170: return "red"
    if h < 22: return "orange"
    if h < 35: return "yellow"
    if h < 85: return "green"
    if h < 115: return "blue"
    return "purple"

def read_board(img):
    board = []
    for r in range(N):
        row = []
        for c in range(N):
            x, y = int((c + .5) * CW), int((r + .5) * CH)
            row.append(identify(img[y - 4:y + 5, x - 4:x + 5]))
        board.append(row)
    return board

def print_board(board):
    for row in board:
        print(" " + " ".join(SYMBOL.get(c, "?") for c in row))

# =========================
# 演算法邏輯
# =========================
def find_matches(board):
    found = set()
    # 檢查水平
    for r in range(N):
        c = 0
        while c < N:
            color = board[r][c]
            end = c
            while end < N and board[r][end] == color:
                end += 1
            if color not in IGNORE and end - c >= 3:
                found.update((r, x) for x in range(c, end))
            c = end
    # 檢查垂直
    for c in range(N):
        r = 0
        while r < N:
            color = board[r][c]
            end = r
            while end < N and board[end][c] == color:
                end += 1
            if color not in IGNORE and end - r >= 3:
                found.update((y, c) for y in range(r, end))
            r = end
    return found

def best_move(board):
    best, best_score = None, 0
    for r in range(N):
        for c in range(N):
            for dr, dc in ((0, 1), (1, 0)):
                r2, c2 = r + dr, c + dc
                if r2 >= N or c2 >= N: continue
                color, other = board[r][c], board[r2][c2]
                if color in IGNORE or other in IGNORE or color == other: continue
                
                test = [row[:] for row in board]
                test[r][c], test[r2][c2] = other, color
                found = find_matches(test)
                
                if (r, c) not in found and (r2, c2) not in found: continue
                
                if len(found) > best_score:
                    best_score = len(found)
                    best = ((r, c), (r2, c2), best_score)
    return best

def board_is_empty(board):
    empty_count = sum(cell == "empty" for row in board for cell in row)
    return empty_count >= N * N * 0.7

def wait_stable(timeout=SETTLE_TIMEOUT):
    deadline = time.time() + timeout
    last, same, board = None, 0, None
    while time.time() < deadline:
        board = read_board(capture())
        sig = tuple(tuple(row) for row in board)
        if sig == last:
            same += 1
            if same >= 2: return board
        else:
            same, last = 0, sig
        time.sleep(0.06)
    return board

# =========================
# 執行模式
# =========================
def calibrate():
    img = capture()
    print_board(read_board(img))
    cv2.imencode(".png", img)[1].tofile("board_debug.png")
    print("已儲存校正截圖至 board_debug.png")

def main():
    print(f"自動消除啟動！{START_DELAY} 秒後開始，按 Q 鍵可隨時停止。")
    time.sleep(START_DELAY)
    
    start, moves, still, last_board = time.time(), 0, 0, None
    
    while time.time() - start < ROUND_TIME:
        if stop_pressed():
            print("手動停止")
            break
            
        board = read_board(capture())
        if board_is_empty(board):
            print("遊戲結束")
            break
            
        if VERBOSE and board != last_board:
            print_board(board)
            last_board = [row[:] for row in board]
            
        move = best_move(board)
        if not move:
            still += 1
            if still >= STILL_LIMIT: break
            time.sleep(0.4)
            continue
            
        still = 0
        a, b, score = move
        print(f"[{time.time()-start:4.1f}s] 交換 {a} <-> {b} (消 {score} 顆)")
        swap(a, b)
        moves += 1
        wait_stable()

    print(f"結束：共交換 {moves} 次")

if __name__ == "__main__":
    mode = sys.argv[1] if len(sys.argv) > 1 else "run"
    if mode == "calibrate": calibrate()
    else: main()

自動消除啟動！5 秒後開始，按 Q 鍵可隨時停止。
 P P P P P P P P
 P P P P P P P P
 P P P P P P P P
 . . P . . . . R
 . . O . . . . R
 R G R P . . W R
 O R P Y . . O R
 P W Y W P P . R
[ 0.0s] 交換 (5, 1) <-> (6, 1) (消 33 顆)
[ 0.6s] 交換 (5, 1) <-> (6, 1) (消 33 顆)
[ 1.1s] 交換 (5, 1) <-> (6, 1) (消 33 顆)
[ 1.6s] 交換 (5, 1) <-> (6, 1) (消 33 顆)
[ 2.1s] 交換 (5, 1) <-> (6, 1) (消 33 顆)
[ 2.6s] 交換 (5, 1) <-> (6, 1) (消 33 顆)
[ 3.1s] 交換 (5, 1) <-> (6, 1) (消 33 顆)
[ 3.6s] 交換 (5, 1) <-> (6, 1) (消 33 顆)
[ 4.1s] 交換 (5, 1) <-> (6, 1) (消 33 顆)
[ 4.6s] 交換 (5, 1) <-> (6, 1) (消 33 顆)
[ 5.1s] 交換 (5, 1) <-> (6, 1) (消 33 顆)
[ 5.6s] 交換 (5, 1) <-> (6, 1) (消 33 顆)
[ 6.1s] 交換 (5, 1) <-> (6, 1) (消 33 顆)
[ 6.6s] 交換 (5, 1) <-> (6, 1) (消 33 顆)
[ 7.1s] 交換 (5, 1) <-> (6, 1) (消 33 顆)
[ 7.6s] 交換 (5, 1) <-> (6, 1) (消 33 顆)
 P P P P P P P P
 P P P P P P P P
 P P P P P P P P
 . . P . . . . R
 . . W . . . . R
 P O O P . . W R
 O G P Y . . O R
 P W Y W P P . R
[ 8.7s] 交換 (5, 0) <-> (6, 0) (消 33 顆)
[ 9.2s] 交換 (5, 0) <-> (6, 0) (消 33 顆)
[ 9.7s] 交換 (5, 0) 

In [4]:
"""
寶石方塊（Bejeweled Blitz）自動消除程式

流程
  1. 截取棋盤畫面（x = 501~814, y = 376~691）
  2. 辨識 8x8 棋盤上每顆寶石的顏色
  3. 找出所有能消除的交換組合，挑選消除數量最多的
  4. 用滑鼠把兩顆寶石拖曳交換
  5. 等棋盤掉落穩定後重複，直到時間到或遊戲結束

使用方式
  python 寶石炸彈.py            開始自動消除
  python 寶石炸彈.py pick       滑鼠指到哪顆寶石就顯示它的顏色（校正用）
  python 寶石炸彈.py calibrate  一次辨識整個盤面，並輸出標註圖 board_debug.png
  python 寶石炸彈.py selftest   不開遊戲，測試判斷邏輯

停止方式
  按 q，或把滑鼠移到螢幕左上角（pyautogui 的安全機制）
"""

from collections import Counter

import ctypes
import sys
import time

import cv2
import numpy as np
import pyautogui
from PIL import ImageGrab

# 讓 pyautogui 與截圖使用相同的實體像素座標（高 DPI 螢幕必備）
try:
    ctypes.windll.shcore.SetProcessDpiAwareness(1)
except Exception:
    pass

# =========================
# 棋盤位置（螢幕座標）
# =========================
LEFT, TOP = 387, 507
RIGHT, BOTTOM = 775, 896
WIDTH, HEIGHT = RIGHT - LEFT, BOTTOM - TOP
N = 8

CW, CH = WIDTH / N, HEIGHT / N

# =========================
# 寶石定位
# =========================
# basex, basey：左上角寶石的中心點（螢幕座標）
basex = int(LEFT + CW / 2)
basey = int(TOP + CH / 2)

# gap：每顆寶石的間距（可給單一數值，或 (水平, 垂直)）
gap = (CW, CH)


def gem_pos(r, c):
    """回傳第 r 列（row）、第 c 行（column）那顆寶石的中心點 (x, y)。"""
    gap_x, gap_y = gap if isinstance(gap, (tuple, list)) else (gap, gap)
    return int(basex + c * gap_x), int(basey + r * gap_y)


# =========================
# 時間與流程設定
# =========================
ROUND_TIME = 65          # Bejeweled Blitz 一局約 60 秒，留一點緩衝
START_DELAY = 5       # 按下執行後，等幾秒才開始動作
STILL_LIMIT = 8          # 連續幾次找不到步可走，就判定遊戲結束
SETTLE_TIMEOUT = 0.05 # 等棋盤穩定的最長時間（秒）
VERBOSE = True           # 是否印出每次辨識到的棋盤

# 不列入消除判斷的格子
IGNORE = {"empty", "unknown"}

SYMBOL = {
    "red": "R", "orange": "O", "yellow": "Y", "green": "G",
    "blue": "B", "purple": "P", "white": "W",
    "empty": ".", "unknown": "?",
}


# =========================
# 截圖
# =========================
def capture():
    img = ImageGrab.grab(
        bbox=(LEFT, TOP, RIGHT, BOTTOM), all_screens=True
    )
    return cv2.cvtColor(np.array(img), cv2.COLOR_RGB2BGR)


def stop_pressed():
    """按 Q 停止（不需額外套件）。"""
    return bool(ctypes.windll.user32.GetAsyncKeyState(ord("Q")) & 0x8000)


# =========================
# 顏色辨識
# =========================
# 門檻值：跑過 calibrate / pick 之後，可依印出的數值微調
MIN_VALUE = 55      # 亮度低於此 → 空格（背景）
MIN_SAT = 45        # 飽和度低於此 → 白色（沒有明顯色彩）
SAMPLE_RADIUS = 8   # 每格中心取樣 17x17，降低反光與邊緣干擾

# 橘色 / 黃色辨識：
# 先用色相區分明顯的橘、黃；落在重疊區時，再用 G/R 比例輔助判斷。
# 黃色通常 G 接近 R（比例較高）；橘色通常 G 明顯低於 R。
# 若實際遊戲配色不同，可執行 calibrate，依印出的 g/r 微調 0.78。
YELLOW_G_RATIO = 0.78

# 色相（OpenCV 的 H 是 0~179）分界
HUE_RED_END = 8
HUE_ORANGE_PURE = 20   # H < 20 → 橘色
HUE_YELLOW_START = 20
HUE_YELLOW_END = 38    # H 20~38 → 再依 G/R 比例分黃、橘
HUE_GREEN_END = 85
HUE_BLUE_END = 135     # 純藍約 H=120，紫約 140 以上

# 取樣方式：kmeans 對反光、陰影、雜訊最穩（測試見 color_methods.py）
COLOR_METHOD = "kmeans"     # 可改成 "median"（較快、但遇到反光較易誤判）


def hsv_median(patch):
    """回傳一小塊畫面（BGR）的 HSV 中位數。"""
    hsv = cv2.cvtColor(patch, cv2.COLOR_BGR2HSV)
    return tuple(int(x) for x in np.median(hsv.reshape(-1, 3), axis=0))


def bgr_median(patch):
    """回傳一小塊畫面（BGR）的 BGR 中位數。"""
    return tuple(int(x) for x in np.median(patch.reshape(-1, 3), axis=0))


def kmeans_color(patch, k=3):
    """用 K-means 把格子分成 k 群，取最大那群當作寶石本身的顏色。

    這樣反光、邊緣、背景只會落在其他小群，不會拉歪顏色。
    回傳 (h, s, v, r, g)。
    """
    data = patch.reshape(-1, 3).astype(np.float32)
    criteria = (cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER, 20, 1.0)
    _, labels, centers = cv2.kmeans(
        data, k, None, criteria, 5, cv2.KMEANS_PP_CENTERS)

    biggest = int(np.argmax(np.bincount(labels.ravel(), minlength=k)))
    pixel = centers[biggest].reshape(1, 1, 3).astype(np.uint8)
    h, s, v = (int(x) for x in cv2.cvtColor(pixel, cv2.COLOR_BGR2HSV)[0, 0])
    b, g, r = (int(x) for x in pixel[0, 0])
    return h, s, v, r, g, b


# 依你提供的遊戲截圖校正出的主要色相中心（OpenCV H：0~179）
# 紅色跨越 0/179，因此使用圓形色相距離計算。
HUE_CENTERS = {
    "red": 176,
    "orange": 13,
    "yellow": 29,
    "green": 64,
    "blue": 106,
    "purple": 150,
}


def hue_distance(a, b):
    """計算 OpenCV 色相環上的最短距離。"""
    return min((int(a) - int(b)) % 180, (int(b) - int(a)) % 180)


def sample_color(patch):
    """用多數像素投票取主要顏色，降低寶石反光、陰影及特效的影響。"""
    hsv = cv2.cvtColor(patch, cv2.COLOR_BGR2HSV)
    pixels = hsv.reshape(-1, 3)

    # 過暗像素多半是格子背景；低彩度亮像素多半是白色反光。
    bright = pixels[pixels[:, 2] >= MIN_VALUE]
    if len(bright) == 0:
        return 0, 0, 0, 0, 0, 0

    chromatic = bright[(bright[:, 1] >= 55) & (bright[:, 2] >= 65)]
    chromatic_ratio = len(chromatic) / max(len(bright), 1)

    # 白色寶石／白色反光：有亮度但缺少穩定的高彩度主色。
    if chromatic_ratio < 0.22:
        med_s = int(np.median(bright[:, 1]))
        med_v = int(np.median(bright[:, 2]))
        return 0, med_s, med_v, 255, 255, 255

    # 每個彩色像素投票給最近的校正色相中心；排除離所有中心太遠的雜訊。
    votes = {name: [] for name in HUE_CENTERS}
    for pixel in chromatic:
        h, s, v = (int(x) for x in pixel)
        distances = {name: hue_distance(h, center)
                     for name, center in HUE_CENTERS.items()}
        nearest = min(distances, key=distances.get)
        if distances[nearest] <= 18:
            votes[nearest].append((h, s, v))

    winner = max(votes, key=lambda name: len(votes[name]))
    selected = votes[winner]

    # 若顏色像素不足或沒有明顯主色，使用全部彩色像素的色相中位數作備援。
    if len(selected) < max(4, int(len(chromatic) * 0.12)):
        selected = [tuple(int(x) for x in p) for p in chromatic]

    selected = np.asarray(selected, dtype=np.uint8)
    h = int(np.median(selected[:, 0]))
    s = int(np.median(selected[:, 1]))
    v = int(np.median(selected[:, 2]))

    # 將代表 HSV 轉成 BGR，再回傳原程式需要的 RGB 欄位。
    representative = np.uint8([[[h, s, v]]])
    b, g, r = (int(x) for x in cv2.cvtColor(
        representative, cv2.COLOR_HSV2BGR)[0, 0])
    return h, s, v, r, g, b


def identify(patch):
    """依校正色相中心辨識寶石，特別分開橘色與黃色。"""
    h, s, v, r, g, b = sample_color(patch)

    if v < MIN_VALUE:
        return "empty"
    if s < MIN_SAT:
        return "white"

    distances = {name: hue_distance(h, center)
                 for name, center in HUE_CENTERS.items()}
    color = min(distances, key=distances.get)

    # 超出合理色相範圍時標記 unknown，避免錯誤顏色進入消除判斷。
    if distances[color] > 18:
        return "unknown"
    return color


def read_board(img):
    """把截圖轉成 8x8 的顏色盤面。"""
    board = []
    for r in range(N):
        row = []
        for c in range(N):
            x = int((c + .5) * CW)
            y = int((r + .5) * CH)
            patch = img[y - SAMPLE_RADIUS:y + SAMPLE_RADIUS + 1,
                        x - SAMPLE_RADIUS:x + SAMPLE_RADIUS + 1]
            row.append(identify(patch))
        board.append(row)
    return board


def print_board(board):
    for row in board:
        print("  " + " ".join(SYMBOL.get(c, "?") for c in row))


def annotate(img, board):
    """把辨識結果畫在截圖上（放大 3 倍），方便肉眼檢查。"""
    out = cv2.resize(img, None, fx=3, fy=3, interpolation=cv2.INTER_NEAREST)

    for i in range(N + 1):
        x = int(i * CW * 3)
        y = int(i * CH * 3)
        cv2.line(out, (x, 0), (x, out.shape[0] - 1), (0, 255, 0), 1)
        cv2.line(out, (0, y), (out.shape[1] - 1, y), (0, 255, 0), 1)

    for r in range(N):
        for c in range(N):
            x = int((c + .5) * CW * 3) - 7
            y = int((r + .5) * CH * 3) + 7
            cv2.putText(out, SYMBOL.get(board[r][c], "?"), (x, y),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 0, 255), 2,
                        cv2.LINE_AA)

    return out


# =========================
# 消除判斷
# =========================
def find_matches(board):
    """回傳所有會被消除的格子座標集合 {(r, c), ...}。"""
    found = set()

    # 水平
    for r in range(N):
        c = 0
        while c < N:
            color = board[r][c]
            end = c
            while end < N and board[r][end] == color:
                end += 1
            if color not in IGNORE and end - c >= 3:
                found.update((r, x) for x in range(c, end))
            c = end

    # 垂直
    for c in range(N):
        r = 0
        while r < N:
            color = board[r][c]
            end = r
            while end < N and board[end][c] == color:
                end += 1
            if color not in IGNORE and end - r >= 3:
                found.update((y, c) for y in range(r, end))
            r = end

    return found


# 舊名稱，方便沿用
matches = find_matches


def move_key(a, b):
    """將交換位置標準化，避免 A<->B 與 B<->A 被當成不同步驟。"""
    return tuple(sorted((tuple(a), tuple(b))))


def best_move(board, excluded_moves=None):
    """回傳 (a, b, score)；排除本盤面已嘗試過的交換位置。"""
    if excluded_moves is None:
        excluded_moves = set()

    best = None
    best_score = 0

    for r in range(N):
        for c in range(N):
            for dr, dc in ((0, 1), (1, 0)):
                r2, c2 = r + dr, c + dc
                if r2 >= N or c2 >= N:
                    continue

                a, b = (r, c), (r2, c2)
                if move_key(a, b) in excluded_moves:
                    continue

                color = board[r][c]
                other = board[r2][c2]
                if color in IGNORE or other in IGNORE or color == other:
                    continue

                test = [row[:] for row in board]
                test[r][c], test[r2][c2] = other, color
                found = find_matches(test)

                # 交換的位置必須真的被消掉，才算是有效步
                if a not in found and b not in found:
                    continue

                score = len(found)
                if score > best_score:
                    best_score = score
                    best = (a, b, score)

    return best


def board_is_empty(board):
    """整個盤面幾乎都是空格 → 遊戲結束（或已離開遊戲）。"""
    empty = sum(cell == "empty" for row in board for cell in row)
    return empty >= N * N * 0.7


def signature(board):
    return tuple(tuple(row) for row in board)


# =========================
# 滑鼠交換
# =========================
def swap(a, b):
    x1, y1 = gem_pos(*a)
    x2, y2 = gem_pos(*b)
    pyautogui.moveTo(x1, y1)
    pyautogui.dragTo(x2, y2, duration=0.12, button="left")


def wait_stable(timeout=SETTLE_TIMEOUT):
    """等畫面連續兩次一樣，代表掉落動畫結束。"""
    deadline = time.time() + timeout
    last = None
    same = 0
    board = None

    while time.time() < deadline:
        board = read_board(capture())
        sig = signature(board)
        if sig == last:
            same += 1
            if same >= 2:
                return board
        else:
            same = 0
            last = sig
        time.sleep(0.06)

    return board


# =========================
# 校正模式
# =========================
def calibrate():
    img = capture()
    board = read_board(img)

    print(f"棋盤範圍：x {LEFT}~{RIGHT}, y {TOP}~{BOTTOM}（{WIDTH}x{HEIGHT}）")
    print("辨識結果（R 紅 O 橙 Y 黃 G 綠 B 藍 P 紫 W 白 . 空）：")
    print_board(board)

    print("\n各格中心 HSV / RGB（黃橘可用 g/r 比例核對）：")
    for r in range(N):
        for c in range(N):
            x = int((c + .5) * CW)
            y = int((r + .5) * CH)
            patch = img[y - SAMPLE_RADIUS:y + SAMPLE_RADIUS + 1,
                        x - SAMPLE_RADIUS:x + SAMPLE_RADIUS + 1]
            h, s, v, rr, g, b = sample_color(patch)
            print(f"  ({r},{c}) -> {board[r][c]:7s} "
                  f"h={h:3d} s={s:3d} v={v:3d} "
                  f"rgb=({rr:3d},{g:3d},{b:3d}) g/r={g / max(rr, 1):.2f}")

    counts = Counter(cell for row in board for cell in row)
    print("\n顏色統計：" + "、".join(
        f"{name} x{n}" for name, n in counts.most_common()))

    if counts.most_common(1)[0][1] >= N * N * 0.9:
        print("！將近全部格子都是同一種顏色，可能畫面不在遊戲上，"
              "或棋盤座標 / 門檻需要調整")

    path = "board_debug.png"
    ok, buf = cv2.imencode(".png", annotate(img, board))
    if ok:
        buf.tofile(path)      # 用 tofile 才支援中文路徑
        print(f"\n標註圖已存成 {path}"
              "（綠線是格子，紅字是辨識結果，可用來核對）")


# =========================
# 單點取樣模式
# =========================
def pick():
    """滑鼠指到哪裡，就即時顯示那一點的顏色，用來校正門檻。"""
    print("把滑鼠移到寶石上，會即時顯示辨識結果（按 q 結束）")
    last = None

    while not stop_pressed():
        x, y = pyautogui.position()

        if LEFT <= x < RIGHT and TOP <= y < BOTTOM:
            rx, ry = x - LEFT, y - TOP
            img = capture()
            patch = img[max(ry - SAMPLE_RADIUS, 0):ry + SAMPLE_RADIUS + 1,
                        max(rx - SAMPLE_RADIUS, 0):rx + SAMPLE_RADIUS + 1]
            h, s, v, rr, g, b = sample_color(patch)
            text = (f"({x},{y}) 第 {int(ry // CH)} 列 第 {int(rx // CW)} 行"
                    f" -> {identify(patch):7s} h={h:3d} s={s:3d} v={v:3d}"
                    f" rgb=({rr:3d},{g:3d},{b:3d}) g/r={g / max(rr, 1):.2f}")
        else:
            text = f"({x},{y}) 不在棋盤範圍內"

        if text != last:
            print(text)
            last = text

        time.sleep(0.15)


# =========================
# 邏輯測試（不需要遊戲畫面）
# =========================
def selftest():
    import random

    rng = random.Random(20260101)
    colors = ["red", "green", "blue", "yellow", "purple", "orange"]

    # 固定亂數種子，找一個「沒有現成三連、但存在有效步」的盤面
    base = None
    for _ in range(5000):
        cand = [[rng.choice(colors) for _ in range(N)] for _ in range(N)]
        if not find_matches(cand) and best_move(cand) is not None:
            base = cand
            break
    assert base is not None, "找不到合適的測試盤面"

    # 橫向三連要被找到
    lined = [row[:] for row in base]
    lined[0][0] = lined[0][1] = lined[0][2] = "red"
    found = find_matches(lined)
    assert {(0, 0), (0, 1), (0, 2)} <= found, "橫向三連沒抓到"

    # 縱向三連要被找到
    lined = [row[:] for row in base]
    lined[0][4] = lined[1][4] = lined[2][4] = "blue"
    found = find_matches(lined)
    assert {(0, 4), (1, 4), (2, 4)} <= found, "縱向三連沒抓到"

    # 一定找得到有效步，而且該步真的能消
    move = best_move(base)
    assert move is not None, "找不到有效步"

    a, b, score = move
    test = [row[:] for row in base]
    test[a[0]][a[1]], test[b[0]][b[1]] = test[b[0]][b[1]], test[a[0]][a[1]]
    found = find_matches(test)
    assert a in found or b in found, "選到的步不會消除"
    assert score == len(found), "分數與實際消除數不一致"

    # 空格過多 → 視為遊戲結束
    empty = [["empty"] * N for _ in range(N)]
    assert board_is_empty(empty)

    # 顏色辨識（BGR 由 RGB 轉換）
    def patch_of(rgb):
        return np.full((9, 9, 3), tuple(reversed(rgb)), dtype=np.uint8)

    cases = [
        ((255, 228, 0), "yellow"),    # 黃：G/R 比例高
        ((255, 210, 0), "yellow"),
        ((255, 190, 0), "orange"),    # 橘：G/R 比例低於分界
        ((255, 154, 0), "orange"),
        ((255, 140, 0), "orange"),
        ((255, 0, 0), "red"),
        ((0, 200, 0), "green"),
        ((0, 0, 255), "blue"),
        ((200, 0, 200), "purple"),
        ((255, 255, 255), "white"),
        ((20, 20, 20), "empty"),
    ]
    for rgb, want in cases:
        got = identify(patch_of(rgb))
        assert got == want, f"rgb={rgb} 應為 {want}，卻判成 {got}"

    print(f"selftest OK（範例最佳步：{a} <-> {b}，可消 {score} 顆）")


# =========================
# 主流程
# =========================
def main():
    print("寶石方塊 自動消除")
    print(f"棋盤範圍：x {LEFT}~{RIGHT}, y {TOP}~{BOTTOM}")
    print(f"{START_DELAY} 秒後開始，請先切換到遊戲畫面")
    print("按 q，或把滑鼠移到左上角可停止")
    time.sleep(START_DELAY)

    start = time.time()
    moves = 0
    still = 0
    last_board = None
    attempted_signature = None
    attempted_moves = set()

    while True:
        if stop_pressed():
            print("手動停止")
            break

        elapsed = time.time() - start
        if elapsed >= ROUND_TIME:
            print(f"時間到（{elapsed:.1f} 秒），停止")
            break

        board = read_board(capture())

        if board_is_empty(board):
            print("盤面幾乎空了，遊戲可能已結束")
            break

        current_signature = signature(board)
        if current_signature != attempted_signature:
            attempted_signature = current_signature
            attempted_moves.clear()
            still = 0

        if VERBOSE and board != last_board:
            print_board(board)
            last_board = [row[:] for row in board]

        move = best_move(board, attempted_moves)

        if move is None:
            still += 1
            if attempted_moves:
                print("這個盤面剩下的有效位置都已經嘗試過，不重複操作，等待棋盤變化。")
            else:
                print(f"找不到可消除的組合（{still}/{STILL_LIMIT}）")

            if still >= STILL_LIMIT:
                print("連續找不到新步可走，判定遊戲可能結束")
                break
            time.sleep(0.15)
            continue

        still = 0
        a, b, score = move
        attempted_moves.add(move_key(a, b))
        print(f"[{elapsed:5.1f}s] 交換 {a} <-> {b}，預計消除 {score} 顆")
        swap(a, b)
        moves += 1
        wait_stable()

    print(f"結束：共交換 {moves} 次，耗時 {time.time() - start:.1f} 秒")


if __name__ == "__main__":
    mode = sys.argv[1] if len(sys.argv) > 1 else "run"
    if mode == "calibrate":
        calibrate()
    elif mode == "pick":
        pick()
    elif mode == "selftest":
        selftest()
    else:
        main()

寶石方塊 自動消除
棋盤範圍：x 387~775, y 507~896
5 秒後開始，請先切換到遊戲畫面
按 q，或把滑鼠移到左上角可停止
  G W W G R R O R
  . . . . . . . .
  O Y W B O G R O
  R . R G R G R .
  R P R P G P R P
  P P P P P P P P
  R R R R R R R R
  O O O O O O O O
[  0.0s] 交換 (0, 6) <-> (0, 7)，預計消除 30 顆
  W G G B P W O P
  R W W R W O O B
  Y P Y O O W P W
  G R P G P O W G
  P B Y G G Y O O
  Y W G Y P G P B
  G W W G R R O R
  Y Y W B O G R O
[  0.6s] 交換 (5, 2) <-> (5, 3)，預計消除 4 顆
  W G G B P W O P
  R W W R W O O B
  Y P Y O O W P W
  G R P . P O W G
  P B Y G G Y O Y
  Y W Y G P G P B
  G W W G R R O R
  Y Y W B O G R O
[  1.3s] 交換 (6, 6) <-> (7, 6)，預計消除 7 顆
  W G G . P W O P
  R W W B W O O B
  Y P Y O O W P W
  G R P B P O W G
  P B Y R G Y O Y
  Y W Y Y P G P B
  G W W G R R R R
  Y Y W B O G O O
[  1.9s] 交換 (0, 0) <-> (1, 0)，預計消除 7 顆
  R G G W W R O W
  W W W B P W O P
  Y P Y O W O P B
  G R P B O . R .
  P B Y Y Y O O G
  Y W Y Y R O P O
  G W W G P G R B
  Y Y W B O G O O
[  2.5s] 交換 (0, 5) <-> (1, 5)，預計消除 9 顆
  R O O W W R 

In [14]:
"""
寶石方塊（Bejeweled Blitz）自動消除程式

流程
  1. 截取棋盤畫面（x = 501~814, y = 376~691）
  2. 辨識 8x8 棋盤上每顆寶石的顏色
  3. 找出所有能消除的交換組合，挑選消除數量最多的
  4. 用滑鼠把兩顆寶石拖曳交換
  5. 等棋盤掉落穩定後重複，直到時間到或遊戲結束

使用方式
  python 寶石炸彈.py            開始自動消除
  python 寶石炸彈.py pick       滑鼠指到哪顆寶石就顯示它的顏色（校正用）
  python 寶石炸彈.py calibrate  一次辨識整個盤面，並輸出標註圖 board_debug.png
  python 寶石炸彈.py selftest   不開遊戲，測試判斷邏輯

停止方式
  按 q，或把滑鼠移到螢幕左上角（pyautogui 的安全機制）
"""

from collections import Counter

import ctypes
import sys
import time

import cv2
import numpy as np
import pyautogui
from PIL import ImageGrab

# 讓 pyautogui 與截圖使用相同的實體像素座標（高 DPI 螢幕必備）
try:
    ctypes.windll.shcore.SetProcessDpiAwareness(1)
except Exception:
    pass

# =========================
# 棋盤位置（螢幕座標）
# =========================
LEFT, TOP = 388, 509
RIGHT, BOTTOM = 777,890
WIDTH, HEIGHT = RIGHT - LEFT, BOTTOM - TOP
N = 8

CW, CH = WIDTH / N, HEIGHT / N

# =========================
# 寶石定位
# =========================
# basex, basey：左上角寶石的中心點（螢幕座標）
basex = int(LEFT + CW / 2)
basey = int(TOP + CH / 2)

# gap：每顆寶石的間距（可給單一數值，或 (水平, 垂直)）
gap = (CW, CH)


def gem_pos(r, c):
    """回傳第 r 列（row）、第 c 行（column）那顆寶石的中心點 (x, y)。"""
    gap_x, gap_y = gap if isinstance(gap, (tuple, list)) else (gap, gap)
    return int(basex + c * gap_x), int(basey + r * gap_y)


# =========================
# 時間與流程設定
# =========================
ROUND_TIME = 65          # Bejeweled Blitz 一局約 60 秒，留一點緩衝
START_DELAY = 5       # 按下執行後，等幾秒才開始動作
STILL_LIMIT = 8          # 連續幾次找不到步可走，就判定遊戲結束
SETTLE_TIMEOUT = 0.05 # 等棋盤穩定的最長時間（秒）
VERBOSE = True           # 是否印出每次辨識到的棋盤

# 不列入消除判斷的格子
IGNORE = {"empty", "unknown"}

SYMBOL = {
    "red": "R", "orange": "O", "yellow": "Y", "green": "G",
    "blue": "B", "purple": "P", "white": "W",
    "empty": ".", "unknown": "?",
}


# =========================
# 截圖
# =========================
def capture():
    img = ImageGrab.grab(
        bbox=(LEFT, TOP, RIGHT, BOTTOM), all_screens=True
    )
    return cv2.cvtColor(np.array(img), cv2.COLOR_RGB2BGR)


def stop_pressed():
    """按 Q 停止（不需額外套件）。"""
    return bool(ctypes.windll.user32.GetAsyncKeyState(ord("Q")) & 0x8000)


# =========================
# 顏色辨識
# =========================
# 門檻值：跑過 calibrate / pick 之後，可依印出的數值微調
MIN_VALUE = 55      # 亮度低於此 → 空格（背景）
MIN_SAT = 45        # 飽和度低於此 → 白色（沒有明顯色彩）
SAMPLE_RADIUS = 8   # 每格中心取樣 17x17，降低反光與邊緣干擾

# 橘色 / 黃色辨識：
# 先用色相區分明顯的橘、黃；落在重疊區時，再用 G/R 比例輔助判斷。
# 黃色通常 G 接近 R（比例較高）；橘色通常 G 明顯低於 R。
# 若實際遊戲配色不同，可執行 calibrate，依印出的 g/r 微調 0.78。
YELLOW_G_RATIO = 0.78

# 色相（OpenCV 的 H 是 0~179）分界
HUE_RED_END = 8
HUE_ORANGE_PURE = 20   # H < 20 → 橘色
HUE_YELLOW_START = 20
HUE_YELLOW_END = 38    # H 20~38 → 再依 G/R 比例分黃、橘
HUE_GREEN_END = 85
HUE_BLUE_END = 135     # 純藍約 H=120，紫約 140 以上

# 取樣方式：kmeans 對反光、陰影、雜訊最穩（測試見 color_methods.py）
COLOR_METHOD = "kmeans"     # 可改成 "median"（較快、但遇到反光較易誤判）


def hsv_median(patch):
    """回傳一小塊畫面（BGR）的 HSV 中位數。"""
    hsv = cv2.cvtColor(patch, cv2.COLOR_BGR2HSV)
    return tuple(int(x) for x in np.median(hsv.reshape(-1, 3), axis=0))


def bgr_median(patch):
    """回傳一小塊畫面（BGR）的 BGR 中位數。"""
    return tuple(int(x) for x in np.median(patch.reshape(-1, 3), axis=0))


def kmeans_color(patch, k=3):
    """用 K-means 把格子分成 k 群，取最大那群當作寶石本身的顏色。

    這樣反光、邊緣、背景只會落在其他小群，不會拉歪顏色。
    回傳 (h, s, v, r, g)。
    """
    data = patch.reshape(-1, 3).astype(np.float32)
    criteria = (cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER, 20, 1.0)
    _, labels, centers = cv2.kmeans(
        data, k, None, criteria, 5, cv2.KMEANS_PP_CENTERS)

    biggest = int(np.argmax(np.bincount(labels.ravel(), minlength=k)))
    pixel = centers[biggest].reshape(1, 1, 3).astype(np.uint8)
    h, s, v = (int(x) for x in cv2.cvtColor(pixel, cv2.COLOR_BGR2HSV)[0, 0])
    b, g, r = (int(x) for x in pixel[0, 0])
    return h, s, v, r, g, b


# 依你提供的遊戲截圖校正出的主要色相中心（OpenCV H：0~179）
# 紅色跨越 0/179，因此使用圓形色相距離計算。
HUE_CENTERS = {
    "red": 176,
    "orange": 13,
    "yellow": 29,
    "green": 64,
    "blue": 106,
    "purple": 150,
}


def hue_distance(a, b):
    """計算 OpenCV 色相環上的最短距離。"""
    return min((int(a) - int(b)) % 180, (int(b) - int(a)) % 180)


def sample_color(patch):
    """用多數像素投票取主要顏色，降低寶石反光、陰影及特效的影響。"""
    hsv = cv2.cvtColor(patch, cv2.COLOR_BGR2HSV)
    pixels = hsv.reshape(-1, 3)

    # 過暗像素多半是格子背景；低彩度亮像素多半是白色反光。
    bright = pixels[pixels[:, 2] >= MIN_VALUE]
    if len(bright) == 0:
        return 0, 0, 0, 0, 0, 0

    chromatic = bright[(bright[:, 1] >= 55) & (bright[:, 2] >= 65)]
    chromatic_ratio = len(chromatic) / max(len(bright), 1)

    # 白色寶石／白色反光：有亮度但缺少穩定的高彩度主色。
    if chromatic_ratio < 0.22:
        med_s = int(np.median(bright[:, 1]))
        med_v = int(np.median(bright[:, 2]))
        return 0, med_s, med_v, 255, 255, 255

    # 每個彩色像素投票給最近的校正色相中心；排除離所有中心太遠的雜訊。
    votes = {name: [] for name in HUE_CENTERS}
    for pixel in chromatic:
        h, s, v = (int(x) for x in pixel)
        distances = {name: hue_distance(h, center)
                     for name, center in HUE_CENTERS.items()}
        nearest = min(distances, key=distances.get)
        if distances[nearest] <= 18:
            votes[nearest].append((h, s, v))

    winner = max(votes, key=lambda name: len(votes[name]))
    selected = votes[winner]

    # 若顏色像素不足或沒有明顯主色，使用全部彩色像素的色相中位數作備援。
    if len(selected) < max(4, int(len(chromatic) * 0.12)):
        selected = [tuple(int(x) for x in p) for p in chromatic]

    selected = np.asarray(selected, dtype=np.uint8)
    h = int(np.median(selected[:, 0]))
    s = int(np.median(selected[:, 1]))
    v = int(np.median(selected[:, 2]))

    # 將代表 HSV 轉成 BGR，再回傳原程式需要的 RGB 欄位。
    representative = np.uint8([[[h, s, v]]])
    b, g, r = (int(x) for x in cv2.cvtColor(
        representative, cv2.COLOR_HSV2BGR)[0, 0])
    return h, s, v, r, g, b


def identify(patch):
    """依校正色相中心辨識寶石，特別分開橘色與黃色。"""
    h, s, v, r, g, b = sample_color(patch)

    if v < MIN_VALUE:
        return "empty"
    if s < MIN_SAT:
        return "white"

    distances = {name: hue_distance(h, center)
                 for name, center in HUE_CENTERS.items()}
    color = min(distances, key=distances.get)

    # 超出合理色相範圍時標記 unknown，避免錯誤顏色進入消除判斷。
    if distances[color] > 18:
        return "unknown"
    return color


def read_board(img):
    """把截圖轉成 8x8 的顏色盤面。"""
    board = []
    for r in range(N):
        row = []
        for c in range(N):
            x = int((c + .5) * CW)
            y = int((r + .5) * CH)
            patch = img[y - SAMPLE_RADIUS:y + SAMPLE_RADIUS + 1,
                        x - SAMPLE_RADIUS:x + SAMPLE_RADIUS + 1]
            row.append(identify(patch))
        board.append(row)
    return board


def print_board(board):
    for row in board:
        print("  " + " ".join(SYMBOL.get(c, "?") for c in row))


def annotate(img, board):
    """把辨識結果畫在截圖上（放大 3 倍），方便肉眼檢查。"""
    out = cv2.resize(img, None, fx=3, fy=3, interpolation=cv2.INTER_NEAREST)

    for i in range(N + 1):
        x = int(i * CW * 3)
        y = int(i * CH * 3)
        cv2.line(out, (x, 0), (x, out.shape[0] - 1), (0, 255, 0), 1)
        cv2.line(out, (0, y), (out.shape[1] - 1, y), (0, 255, 0), 1)

    for r in range(N):
        for c in range(N):
            x = int((c + .5) * CW * 3) - 7
            y = int((r + .5) * CH * 3) + 7
            cv2.putText(out, SYMBOL.get(board[r][c], "?"), (x, y),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 0, 255), 2,
                        cv2.LINE_AA)

    return out


# =========================
# 消除判斷
# =========================
def find_matches(board):
    """回傳所有會被消除的格子座標集合 {(r, c), ...}。"""
    found = set()

    # 水平
    for r in range(N):
        c = 0
        while c < N:
            color = board[r][c]
            end = c
            while end < N and board[r][end] == color:
                end += 1
            if color not in IGNORE and end - c >= 3:
                found.update((r, x) for x in range(c, end))
            c = end

    # 垂直
    for c in range(N):
        r = 0
        while r < N:
            color = board[r][c]
            end = r
            while end < N and board[end][c] == color:
                end += 1
            if color not in IGNORE and end - r >= 3:
                found.update((y, c) for y in range(r, end))
            r = end

    return found


# 舊名稱，方便沿用
matches = find_matches


def move_key(a, b):
    """將交換位置標準化，避免 A<->B 與 B<->A 被當成不同步驟。"""
    return tuple(sorted((tuple(a), tuple(b))))


def best_move(board, excluded_moves=None):
    """回傳 (a, b, score)；排除本盤面已嘗試過的交換位置。"""
    if excluded_moves is None:
        excluded_moves = set()

    best = None
    best_score = 0

    for r in range(N):
        for c in range(N):
            for dr, dc in ((0, 1), (1, 0)):
                r2, c2 = r + dr, c + dc
                if r2 >= N or c2 >= N:
                    continue

                a, b = (r, c), (r2, c2)
                if move_key(a, b) in excluded_moves:
                    continue

                color = board[r][c]
                other = board[r2][c2]
                if color in IGNORE or other in IGNORE or color == other:
                    continue

                test = [row[:] for row in board]
                test[r][c], test[r2][c2] = other, color
                found = find_matches(test)

                # 交換的位置必須真的被消掉，才算是有效步
                if a not in found and b not in found:
                    continue

                score = len(found)
                if score > best_score:
                    best_score = score
                    best = (a, b, score)

    return best


def board_is_empty(board):
    """整個盤面幾乎都是空格 → 遊戲結束（或已離開遊戲）。"""
    empty = sum(cell == "empty" for row in board for cell in row)
    return empty >= N * N * 0.7


def signature(board):
    return tuple(tuple(row) for row in board)


# =========================
# 滑鼠交換
# =========================
def swap(a, b):
    x1, y1 = gem_pos(*a)
    x2, y2 = gem_pos(*b)
    pyautogui.moveTo(x1, y1)
    pyautogui.dragTo(x2, y2, duration=0.12, button="left")


def wait_stable(timeout=SETTLE_TIMEOUT):
    """等畫面連續兩次一樣，代表掉落動畫結束。"""
    deadline = time.time() + timeout
    last = None
    same = 0
    board = None

    while time.time() < deadline:
        board = read_board(capture())
        sig = signature(board)
        if sig == last:
            same += 1
            if same >= 2:
                return board
        else:
            same = 0
            last = sig
        time.sleep(0.06)

    return board


# =========================
# 校正模式
# =========================
def calibrate():
    img = capture()
    board = read_board(img)

    print(f"棋盤範圍：x {LEFT}~{RIGHT}, y {TOP}~{BOTTOM}（{WIDTH}x{HEIGHT}）")
    print("辨識結果（R 紅 O 橙 Y 黃 G 綠 B 藍 P 紫 W 白 . 空）：")
    print_board(board)

    print("\n各格中心 HSV / RGB（黃橘可用 g/r 比例核對）：")
    for r in range(N):
        for c in range(N):
            x = int((c + .5) * CW)
            y = int((r + .5) * CH)
            patch = img[y - SAMPLE_RADIUS:y + SAMPLE_RADIUS + 1,
                        x - SAMPLE_RADIUS:x + SAMPLE_RADIUS + 1]
            h, s, v, rr, g, b = sample_color(patch)
            print(f"  ({r},{c}) -> {board[r][c]:7s} "
                  f"h={h:3d} s={s:3d} v={v:3d} "
                  f"rgb=({rr:3d},{g:3d},{b:3d}) g/r={g / max(rr, 1):.2f}")

    counts = Counter(cell for row in board for cell in row)
    print("\n顏色統計：" + "、".join(
        f"{name} x{n}" for name, n in counts.most_common()))

    if counts.most_common(1)[0][1] >= N * N * 0.9:
        print("！將近全部格子都是同一種顏色，可能畫面不在遊戲上，"
              "或棋盤座標 / 門檻需要調整")

    path = "board_debug.png"
    ok, buf = cv2.imencode(".png", annotate(img, board))
    if ok:
        buf.tofile(path)      # 用 tofile 才支援中文路徑
        print(f"\n標註圖已存成 {path}"
              "（綠線是格子，紅字是辨識結果，可用來核對）")


# =========================
# 單點取樣模式
# =========================
def pick():
    """滑鼠指到哪裡，就即時顯示那一點的顏色，用來校正門檻。"""
    print("把滑鼠移到寶石上，會即時顯示辨識結果（按 q 結束）")
    last = None

    while not stop_pressed():
        x, y = pyautogui.position()

        if LEFT <= x < RIGHT and TOP <= y < BOTTOM:
            rx, ry = x - LEFT, y - TOP
            img = capture()
            patch = img[max(ry - SAMPLE_RADIUS, 0):ry + SAMPLE_RADIUS + 1,
                        max(rx - SAMPLE_RADIUS, 0):rx + SAMPLE_RADIUS + 1]
            h, s, v, rr, g, b = sample_color(patch)
            text = (f"({x},{y}) 第 {int(ry // CH)} 列 第 {int(rx // CW)} 行"
                    f" -> {identify(patch):7s} h={h:3d} s={s:3d} v={v:3d}"
                    f" rgb=({rr:3d},{g:3d},{b:3d}) g/r={g / max(rr, 1):.2f}")
        else:
            text = f"({x},{y}) 不在棋盤範圍內"

        if text != last:
            print(text)
            last = text

        time.sleep(0.15)


# =========================
# 邏輯測試（不需要遊戲畫面）
# =========================
def selftest():
    import random

    rng = random.Random(20260101)
    colors = ["red", "green", "blue", "yellow", "purple", "orange"]

    # 固定亂數種子，找一個「沒有現成三連、但存在有效步」的盤面
    base = None
    for _ in range(5000):
        cand = [[rng.choice(colors) for _ in range(N)] for _ in range(N)]
        if not find_matches(cand) and best_move(cand) is not None:
            base = cand
            break
    assert base is not None, "找不到合適的測試盤面"

    # 橫向三連要被找到
    lined = [row[:] for row in base]
    lined[0][0] = lined[0][1] = lined[0][2] = "red"
    found = find_matches(lined)
    assert {(0, 0), (0, 1), (0, 2)} <= found, "橫向三連沒抓到"

    # 縱向三連要被找到
    lined = [row[:] for row in base]
    lined[0][4] = lined[1][4] = lined[2][4] = "blue"
    found = find_matches(lined)
    assert {(0, 4), (1, 4), (2, 4)} <= found, "縱向三連沒抓到"

    # 一定找得到有效步，而且該步真的能消
    move = best_move(base)
    assert move is not None, "找不到有效步"

    a, b, score = move
    test = [row[:] for row in base]
    test[a[0]][a[1]], test[b[0]][b[1]] = test[b[0]][b[1]], test[a[0]][a[1]]
    found = find_matches(test)
    assert a in found or b in found, "選到的步不會消除"
    assert score == len(found), "分數與實際消除數不一致"

    # 空格過多 → 視為遊戲結束
    empty = [["empty"] * N for _ in range(N)]
    assert board_is_empty(empty)

    # 顏色辨識（BGR 由 RGB 轉換）
    def patch_of(rgb):
        return np.full((9, 9, 3), tuple(reversed(rgb)), dtype=np.uint8)

    cases = [
        ((255, 228, 0), "yellow"),    # 黃：G/R 比例高
        ((255, 210, 0), "yellow"),
        ((255, 190, 0), "orange"),    # 橘：G/R 比例低於分界
        ((255, 154, 0), "orange"),
        ((255, 140, 0), "orange"),
        ((255, 0, 0), "red"),
        ((0, 200, 0), "green"),
        ((0, 0, 255), "blue"),
        ((200, 0, 200), "purple"),
        ((255, 255, 255), "white"),
        ((20, 20, 20), "empty"),
    ]
    for rgb, want in cases:
        got = identify(patch_of(rgb))
        assert got == want, f"rgb={rgb} 應為 {want}，卻判成 {got}"

    print(f"selftest OK（範例最佳步：{a} <-> {b}，可消 {score} 顆）")


# =========================
# 主流程
# =========================
def main():
    print("寶石方塊 自動消除")
    print(f"棋盤範圍：x {LEFT}~{RIGHT}, y {TOP}~{BOTTOM}")
    print(f"{START_DELAY} 秒後開始，請先切換到遊戲畫面")
    print("按 q，或把滑鼠移到左上角可停止")
    time.sleep(START_DELAY)

    start = time.time()
    moves = 0
    still = 0
    last_board = None
    attempted_signature = None
    attempted_moves = set()

    while True:
        if stop_pressed():
            print("手動停止")
            break

        elapsed = time.time() - start
        if elapsed >= ROUND_TIME:
            print(f"時間到（{elapsed:.1f} 秒），停止")
            break

        board = read_board(capture())

        if board_is_empty(board):
            print("盤面幾乎空了，遊戲可能已結束")
            break

        current_signature = signature(board)
        if current_signature != attempted_signature:
            attempted_signature = current_signature
            attempted_moves.clear()
            still = 0

        if VERBOSE and board != last_board:
            print_board(board)
            last_board = [row[:] for row in board]

        move = best_move(board, attempted_moves)

        if move is None:
            still += 1
            if attempted_moves:
                print("這個盤面剩下的有效位置都已經嘗試過，不重複操作，等待棋盤變化。")
            else:
                print(f"找不到可消除的組合（{still}/{STILL_LIMIT}）")

            if still >= STILL_LIMIT:
                print("連續找不到新步可走，判定遊戲可能結束")
                break
            time.sleep(0.15)
            continue

        still = 0
        a, b, score = move
        attempted_moves.add(move_key(a, b))
        print(f"[{elapsed:5.1f}s] 交換 {a} <-> {b}，預計消除 {score} 顆")
        swap(a, b)
        moves += 1
        wait_stable()

    print(f"結束：共交換 {moves} 次，耗時 {time.time() - start:.1f} 秒")


if __name__ == "__main__":
    mode = sys.argv[1] if len(sys.argv) > 1 else "run"
    if mode == "calibrate":
        calibrate()
    elif mode == "pick":
        pick()
    elif mode == "selftest":
        selftest()
    else:
        main()

寶石方塊 自動消除
棋盤範圍：x 388~777, y 509~890
5 秒後開始，請先切換到遊戲畫面
按 q，或把滑鼠移到左上角可停止
  P G Y G Y R P G
  . . . . . . . .
  . . . . . . . .
  R . R . R . R .
  R P R P R P R P
  P P P P P P P P
  P R P R P R P R
  R R R R R R R R
[  0.0s] 交換 (4, 0) <-> (4, 1)，預計消除 18 顆
  W R B B W W B O
  O Y R R P Y R R
  O Y R R W Y R R
  R . R . R . R .
  P O O G G P Y P
  P P P P P P P P
  O Y P Y P O Y R
  P G Y G Y R P G
[  0.4s] 交換 (6, 2) <-> (7, 2)，預計消除 17 顆
  B R Y G O P Y Y
  R P R Y G O B R
  B Y Y W Y B R W
  W R B B W W B Y
  Y Y R R G Y R R
  P Y Y G R W Y B
  O Y Y Y P O P R
  P G P G Y R P G
[  0.9s] 交換 (1, 3) <-> (2, 3)，預計消除 9 顆
  B . . . O P Y Y
  R R Y G G O B R
  B P R Y Y B R W
  W Y Y W W W B Y
  Y R B B G Y R R
  P Y R R R W Y B
  P O P G P O P R
  P G P G Y R P G
[  1.4s] 交換 (1, 2) <-> (2, 2)，預計消除 15 顆
  G P G G O B Y Y
  B W . . . P B R
  R R Y W O O R W
  B P Y G G B B Y
  W Y R O G Y R R
  Y R Y B W W Y B
  P Y B G P O P R
  P G P G Y R P G
[  1.9s] 交換 (4, 1) <-> (4, 2)，預計消除 4 顆
  G P P G R 

In [27]:

import pyautogui
import time

# --- 使用前唯一步驟：請手動把遊戲盤面，拖移到螢幕的左上角（靠近邊緣） ---
START_X = 407   # 遊戲盤面第一顆寶石的 X 座標
START_Y = 523  # 遊戲盤面第一顆寶石的 Y 座標
GRID = 52      # 每顆寶石的格子大小（像素）

print("請在 3 秒內切換到遊戲視窗...")
time.sleep(3)

print("外掛啟動！開始全盤瘋狂上下左右亂滑...")
# 遍歷 8x8 所有的格子，強迫它們往右、往下和周圍交換
for row in range(8):
    for col in range(8):
        x = START_X + (col * GRID)
        y = START_Y + (row * GRID)
        
        # 往右滑一下
        pyautogui.moveTo(x, y)
        pyautogui.dragTo(x + GRID, y, duration=0.02)
        
        # 往下滑一下
        pyautogui.moveTo(x, y)
        pyautogui.dragTo(x, y + GRID, duration=0.02)

print("亂滑完成！")

請在 3 秒內切換到遊戲視窗...
外掛啟動！開始全盤瘋狂上下左右亂滑...
亂滑完成！


In [35]:
import argparse
import ctypes
from ctypes import wintypes
import logging
import operator
from pathlib import Path
import time

import cv2
import mss
import numpy as np

__file__ = "blitz.py"
SIZE = 8
COLORS = "ROYGBPW"
F2, F8 = 0x71, 0x77
LOG = logging.getLogger(__name__)
LOG_PATH = Path(__file__).with_name("blitz-last-run.log")


def read_board(frame):
    hsv = cv2.cvtColor(cv2.resize(frame, (256, 256)), cv2.COLOR_BGR2HSV)
    tiles = hsv.reshape(8, 32, 8, 32, 3).transpose(0, 2, 1, 3, 4)
    hue, sat, value = tiles[:, :, 7:25, 7:25, :].transpose(4, 0, 1, 2, 3)
    bright = (sat >= 100) & (value >= 95)
    masks = (
        bright & ((hue <= 8) | (hue >= 172)),
        bright & (hue >= 9) & (hue <= 21),
        bright & (hue >= 22) & (hue <= 38),
        bright & (hue >= 39) & (hue <= 82),
        bright & (hue >= 83) & (hue <= 125),
        bright & (hue >= 126) & (hue <= 171),
        (sat <= 80) & (value >= 145),
    )
    counts = np.stack([mask.sum(axis=(2, 3)) for mask in masks])
    ranked = np.sort(counts, axis=0)
    clear = (ranked[-1] >= 100) & (ranked[-1] - ranked[-2] >= 55)
    return tuple(
        tuple(COLORS[color] if valid else None for color, valid in zip(row, good))
        for row, good in zip(counts.argmax(axis=0), clear)
    )


def timer_mask(frame):
    hue, sat, value = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV).transpose(2, 0, 1)
    return (hue >= 83) & (hue <= 107) & (sat >= 100) & (value >= 100)


def find_board(frame):
    _, frame_w = frame.shape[:2]
    mask = timer_mask(frame).astype(np.uint8)
    _, _, components, _ = cv2.connectedComponentsWithStats(mask)
    for x, y, length, thickness, area in sorted(components[1:], key=operator.itemgetter(2), reverse=True):
        if length < 150 or not 14 * thickness <= length <= 29 * thickness:
            continue
        if area < 0.7 * length * thickness:
            continue
        side = round(length * 1.013)
        left = int(x) - round(side * 0.008)
        top = int(y) - round(side * 0.018) - side
        if left < 0 or top < 0 or left + side > frame_w:
            continue
        board_frame = frame[top:top + side, left:left + side]
        gems = [gem for row in read_board(board_frame) for gem in row if gem is not None]
        if len(gems) >= 60 and len(set(gems)) >= 5:
            return left, top, side, side
    return None


def runs(board):
    found = []
    for vertical in (False, True):
        for line in range(SIZE):
            cells = [(i, line) if vertical else (line, i) for i in range(SIZE)]
            start = 0
            while start < SIZE:
                row, col = cells[start]
                color = board[row][col]
                end = start + 1
                while end < SIZE:
                    er, ec = cells[end]
                    if board[er][ec] != color:
                        break
                    end += 1
                if color is not None and end - start >= 3:
                    found.append(set(cells[start:end]))
                start = end
    return found


def predict_value(board, matches, can_predict_cascades):
    value = 0
    for step in range(4 if can_predict_cascades else 1):
        if not matches:
            break
        removed = set.union(*matches)
        longest = max(map(len, matches))
        intersects = sum(map(len, matches)) > len(removed)
        special_bonus = 12 if longest >= 5 else 8 if intersects else 4 if longest == 4 else 0
        value += (2 * len(removed) + special_bonus) / (step + 1)
        for col in range(SIZE):
            kept = [board[row][col] for row in range(SIZE) if (row, col) not in removed]
            for row, gem in enumerate([None] * (SIZE - len(kept)) + kept):
                board[row][col] = gem
        matches = runs(board)
    return value


def best_move(board):
    best = None
    best_value = 0
    complete = all(None not in row for row in board)
    for row in range(SIZE):
        for col in range(SIZE):
            source = board[row][col]
            if source is None:
                continue
            for r, c in ((row + 1, col), (row, col + 1)):
                if r >= SIZE or c >= SIZE or board[r][c] in (source, None):
                    continue
                trial = [list(line) for line in board]
                trial[row][col], trial[r][c] = trial[r][c], trial[row][col]
                matches = runs(trial)
                if not any((row, col) in run or (r, c) in run for run in matches):
                    continue
                value = predict_value(trial, matches, complete)
                if value > best_value:
                    best = ((row, col), (r, c))
                    best_value = value
    return best


class Windows:
    def __init__(self):
        self.user = ctypes.WinDLL("user32", use_last_error=True)
        self.user.SetProcessDPIAware()
        self.user.GetForegroundWindow.restype = wintypes.HWND
        self.user.GetWindowTextW.argtypes = [wintypes.HWND, wintypes.LPWSTR, ctypes.c_int]
        self.user.GetWindowRect.argtypes = [wintypes.HWND, ctypes.POINTER(wintypes.RECT)]
        self.user.GetAsyncKeyState.argtypes = [ctypes.c_int]
        self.user.GetAsyncKeyState.restype = ctypes.c_short
        self.user.SetCursorPos.argtypes = [ctypes.c_int, ctypes.c_int]
        self.user.mouse_event.argtypes = [wintypes.DWORD] * 4 + [ctypes.c_size_t]

    def pressed(self, key):
        return bool(self.user.GetAsyncKeyState(key) & 0x8000)

    def check(self):
        if self.pressed(F8):
            raise KeyboardInterrupt

    def wait_in_game(self, window=None):
        while True:
            while not self.pressed(F2):
                self.check()
                time.sleep(0.015)
            while self.pressed(F2):
                self.check()
                time.sleep(0.015)
            self.check()
            active = self.game_window()
            if active is not None and (window is None or active == window):
                return active
            LOG.warning("這次 F2 已略過。請切回原本的 Chrome 遊戲分頁再按 F2。")

    def game_window(self):
        window = self.user.GetForegroundWindow()
        title = ctypes.create_unicode_buffer(512)
        self.user.GetWindowTextW(window, title, len(title))
        if "bejeweled blitz" not in title.value.lower() or "chrome" not in title.value.lower():
            return None
        return window

    def window_region(self, window, desktop):
        bounds = wintypes.RECT()
        if not self.user.GetWindowRect(window, ctypes.byref(bounds)):
            raise ctypes.WinError(ctypes.get_last_error())
        left = max(bounds.left, desktop["left"])
        top = max(bounds.top, desktop["top"])
        right = min(bounds.right, desktop["left"] + desktop["width"])
        bottom = min(bounds.bottom, desktop["top"] + desktop["height"])
        if right <= left or bottom <= top:
            raise ValueError("Chrome 遊戲視窗在螢幕範圍外。")
        return dict(left=left, top=top, width=right - left, height=bottom - top)

    def move_cursor(self, point):
        if not self.user.SetCursorPos(*point):
            raise ctypes.WinError(ctypes.get_last_error())

    def swap(self, window, source, target):
        self.check()
        if self.game_window() != window:
            return False
        self.move_cursor(source)
        self.check()
        if self.game_window() != window:
            return False
        sx, sy = source
        tx, ty = target
        self.user.mouse_event(0x0002, 0, 0, 0, 0)
        try:
            for step in range(1, 6):
                self.check()
                x = round(sx + (tx - sx) * step / 5)
                y = round(sy + (ty - sy) * step / 5)
                self.move_cursor((x, y))
                time.sleep(0.008)
        finally:
            self.user.mouse_event(0x0004, 0, 0, 0, 0)
        return True


def cell_center(region, rect, cell):
    left, top, width, height = rect
    row, col = cell
    x = region["left"] + left + round((col + 0.5) * width / SIZE)
    y = region["top"] + top + round((row + 0.5) * height / SIZE)
    return x, y


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--delay", type=float, default=0.05)
    parser.add_argument("--debug", action="store_true")
    args, _ = parser.parse_known_args()

    logging.basicConfig(
        level=logging.DEBUG if args.debug else logging.INFO,
        format="%(asctime)s [%(levelname)s] %(message)s",
        handlers=[logging.FileHandler(LOG_PATH, encoding="utf-8"), logging.StreamHandler()],
    )

    win = Windows()
    LOG.info("程式已啟動！切到 Chrome 遊戲分頁並開始遊戲後按 [F2] 開始，按 [F8] 結束。")

    try:
        with mss.mss() as sct:
            desktop, *_ = sct.monitors
            while True:
                window = win.wait_in_game()
                LOG.info("偵測到遊戲視窗，正在定位棋盤...")
                region = win.window_region(window, desktop)
                board_rect = None

                while board_rect is None:
                    win.check()
                    if win.game_window() != window:
                        break
                    raw = np.array(sct.grab(region))[:, :, :3]
                    board_rect = find_board(raw)
                    time.sleep(0.05)

                if board_rect is None:
                    continue

                bx, by, bw, bh = board_rect
                LOG.info("成功定位棋盤，開始自動配對！")
                board_region = dict(
                    left=region["left"] + bx,
                    top=region["top"] + by,
                    width=bw,
                    height=bh,
                )

                empty_frames = 0
                while True:
                    win.check()
                    if win.game_window() != window:
                        LOG.info("遊戲視窗已離開前景，暫停操作。若要繼續請切回視窗並按 F2。")
                        break

                    board_img = np.array(sct.grab(board_region))[:, :, :3]
                    move = best_move(read_board(board_img))

                    if move is None:
                        empty_frames += 1
                        if empty_frames > 60:
                            LOG.info("回合可能已結束，等待下一次按 F2。")
                            break
                        time.sleep(0.03)
                        continue

                    empty_frames = 0
                    src_cell, dst_cell = move
                    src_pt = cell_center(region, board_rect, src_cell)
                    dst_pt = cell_center(region, board_rect, dst_cell)
                    if win.swap(window, src_pt, dst_pt):
                        time.sleep(args.delay)

    except KeyboardInterrupt:
        LOG.info("偵測到 F8 或手動中斷，程式已結束。")


if __name__ == "__main__":
    main()

2026-11-10 02:07:57,497 [INFO] 程式已啟動！切到 Chrome 遊戲分頁並開始遊戲後按 [F2] 開始，按 [F8] 結束。
C:\Users\呂翊楷\AppData\Local\Temp\ipykernel_73536\2369692038.py:237: DeprecationWarning: mss.mss is deprecated and will be removed in a future release; use mss.MSS instead
  with mss.mss() as sct:
2026-11-10 02:08:11,388 [INFO] 偵測到 F8 或手動中斷，程式已結束。
